# Étape 1 - Chargement et nettoyage des données

Dans cette étape on récupère les données brutes des restaurants et on les transforme
en un jeu propre, prêt pour la modélisation.

## Ce que fait le pré-traitement

Il se déroule en 4 briques, déjà codées dans `foodcast/domain/` :

| Fonction | Rôle |
|---|---|
| `extract` | lit les fichiers CSV hebdomadaires d'un restaurant sur un intervalle de semaines |
| `clean` | nettoie : noms de colonnes en minuscules, `order_date` en datetime, calcule `cash_in` (montant par commande), supprime les colonnes inutiles, trie par date |
| `merge` | fusionne restaurant_1 + restaurant_2 en un seul dataframe (la chaîne) |
| `resample` | ré-échantillonne à la maille **1 heure** (somme du `cash_in` par heure) |

Ces 4 briques sont enchaînées par une fonction maître unique : **`etl`**.

## 1. Importer les librairies

In [ ]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

## 2. Regarder le code de `etl`

`??` affiche la signature, la docstring **et** le code source complet.

In [ ]:
etl??

## 3. Extraire un jeu de données pré-traité (semaines 197 à 200)

Signature : `etl(data_dir, start_week, end_week)`.

- `data_dir` : le dossier des données, disponible dans `settings.DATA_DIR`
- `start_week` / `end_week` : numéros de semaine, **inclus** tous les deux

In [ ]:
df = etl(settings.DATA_DIR, 197, 200)
df.head(20)

Le dataframe obtenu a une ligne par heure, avec deux colonnes :

- `order_date` : l'horodatage (début de l'heure)
- `cash_in` : le chiffre d'affaires encaissé pendant cette heure (les deux restaurants réunis)

In [ ]:
df.shape

In [ ]:
df.describe()

## 4. Tracer le chiffre d'affaires en fonction du temps

On utilise `plotly` : un objet `go.Figure()` auquel on ajoute une trace `go.Scatter`
(x = les dates, y = le chiffre d'affaires).

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=df['order_date'], y=df['cash_in'], name='cash-in')
)
fig.update_layout(
    title='Cash-in',
    xaxis_title='date',
    yaxis_title='dollars',
    font=dict(family='Computer Modern', size=18, color='#7f7f7f'),
)
fig.show()

**Ce qu'on observe :** une forte saisonnalité *journalière* (pics le soir, creux la nuit)
et *hebdomadaire* (week-ends plus chargés). C'est exactement ce que le feature engineering
de l'étape suivante va encoder.

➡️ Étape suivante : `02_feature_engineering_offline.ipynb`